In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
import matplotlib.pyplot as plt


In [24]:
df = pd.read_excel('exerc7.xlsx')
df = df.set_index('Propriedade')
df = df.rename(columns={1:'Liga 1',2:'Liga 2',3:'Liga 3',4:'Liga 4',5:'Liga 5'})
df

,Liga 1,Liga 2,Liga 3,Liga 4,Liga 5
Propriedade,,,,,
estanho,60,25,45,20,50
zinco,10,15,45,50,40
chumbo,30,20,25,24,10


In [136]:
limites = {
    'estanho':0.40,
    'zinco':0.35,
    'chumbo':0.25
}

In [130]:
custos = [22,20,25,24,27]
dicio_custo={}
j=0
for i in df.columns:
    dicio_custo[i] = custos[j]
    j=j+1

dicio_custo

{'Liga 1': 22, 'Liga 2': 20, 'Liga 3': 25, 'Liga 4': 24, 'Liga 5': 27}

In [112]:
dicio = {}
for c in df.columns:
    for i in df.index:
        dicio[c] = {a:df[c][a] for a in df.index }
    

In [115]:
dicio

{'Liga 1': {'estanho': np.int64(60),
  'zinco': np.int64(10),
  'chumbo': np.int64(30)},
 'Liga 2': {'estanho': np.int64(25),
  'zinco': np.int64(15),
  'chumbo': np.int64(20)},
 'Liga 3': {'estanho': np.int64(45),
  'zinco': np.int64(45),
  'chumbo': np.int64(25)},
 'Liga 4': {'estanho': np.int64(20),
  'zinco': np.int64(50),
  'chumbo': np.int64(24)},
 'Liga 5': {'estanho': np.int64(50),
  'zinco': np.int64(40),
  'chumbo': np.int64(10)}}

In [167]:
model = pyo.ConcreteModel()

#Definindo Set
model.ligas = pyo.Set(initialize = df.columns)
model.propriedades = pyo.Set(initialize = df.index)

#variaveis
model.x = pyo.Var(model.ligas, bounds=(0,1), domain=pyo.NonNegativeReals)
# model.y = pyo.Var(model.propriedades,bounds=(0,None), domain=pyo.NonNegativeReals)

#Objetivo
def objetivo(model):

    return sum(model.x[l] * dicio_custo[l] for l in model.ligas)
model.objetivo = pyo.Objective(rule=objetivo, sense=pyo.minimize)

# Restrições
def composicao_total(model,l):
    
    return sum(model.x[l] for l in model.ligas) == 1
model.composicao_total = pyo.Constraint(rule=composicao_total)

def restricao_propriedade(model,p):

    return sum(model.x[l] * dicio[l][p]/100 for l in model.ligas) == limites[p]
model.restricao_propriedade = pyo.Constraint(model.propriedades, rule=restricao_propriedade)



In [168]:
model.pprint()

2 Set Declarations
    ligas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {'Liga 1', 'Liga 2', 'Liga 3', 'Liga 4', 'Liga 5'}
    propriedades : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'estanho', 'zinco', 'chumbo'}

1 Var Declarations
    x : Size=5, Index=ligas
        Key    : Lower : Value : Upper : Fixed : Stale : Domain
        Liga 1 :     0 :  None :     1 : False :  True : NonNegativeReals
        Liga 2 :     0 :  None :     1 : False :  True : NonNegativeReals
        Liga 3 :     0 :  None :     1 : False :  True : NonNegativeReals
        Liga 4 :     0 :  None :     1 : False :  True : NonNegativeReals
        Liga 5 :     0 :  None :     1 : False :  True : NonNegativeReals

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   Tr

In [169]:
opt = SolverFactory('cplex')
results = opt.solve(model, tee=True, warmstart=True)



Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp4evfbjj7.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpc6qkp8eb.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Not a mixed integer problem.
No file read.
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpc6qkp8eb.pyomo.lp
Objective sense      : Minimize
Variables            :       5  [Box: 5]
Objective nonzeros   :       5
Linear constraints   :       4  [Equal: 4]
  Nonzeros           :      20
  RHS nonzeros       :       4

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objecti

In [184]:
for i in model.ligas:
    print(f'% da liga {i}: ',model.x[i].value)
print('Custo Total: R$: ',pyo.value(model.objetivo))

% da liga Liga 1:  0.18661971830985924
% da liga Liga 2:  0.14788732394366197
% da liga Liga 3:  0.47183098591549294
% da liga Liga 4:  0.19366197183098582
% da liga Liga 5:  0.0
Custo Total: R$:  23.507042253521128
